# Thư viện OpenCV

In [ ]:
! pip install opencv-python
! pip install paddleocr==2.8.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
import os
from google.colab.patches import cv2_imshow
ROOT = '/content/drive/MyDrive/OCR/'
file_name = 'poem.jpg'
path = os.path.join(ROOT, file_name)
image = cv2.imread(path)

In [ ]:
cv2_imshow(image)

In [ ]:
#Chuyển hệ màu
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

In [ ]:
#Làm mờ và khử nhiễu (sử dụng khi ảnh nhiễu nặng)
# blurrede = cv2.GaussianBlur(gray, (5,5), 0)
denoised = cv2.medianBlur(gray, 3)

In [ ]:
import numpy as np

In [ ]:
#Deskew_xoay ảnh
_, bw = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
coords = np.column_stack(np.where(bw > 0))
angle = cv2.minAreaRect(coords)[-1]

if angle < -45:
    angle = -(90 + angle)
else:
    angle = -angle

(h, w) = image.shape[:2]
center = (w // 2, h // 2)
M = cv2.getRotationMatrix2D(center, angle, 1.0)
deskewed = cv2.warpAffine(gray, M, (w, h))

In [ ]:
#auto-crop (cắt viền thừa)
_, thresh = cv2.threshold(
    deskewed, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

coords = cv2.findNonZero(thresh)

if coords is not None:
    x, y, w, h = cv2.boundingRect(coords)
    cropped = deskewed[y:y+h, x:x+w]
else:
    # fallback: giữ nguyên ảnh
    cropped = deskewed

if len(cropped.shape) == 3:
    cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

In [ ]:
#Xử lý nền: giấy ố vàng, nền không đều + không làm với chữ màu
normalized = cv2.adaptiveThreshold(cropped, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

In [ ]:
#Làm rõ chữ sharpen
kernel = np.array([[0, -1, 0],
                   [-1, 5,-1],
                   [0, -1, 0]])
sharpened = cv2.filter2D(normalized, -1, kernel)

# OCR

In [ ]:
from paddleocr import PaddleOCR

# Khởi tạo PaddleOCR
ocr = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    show_log=False
)

print("PaddleOCR đã sẵn sàng!")

PaddleOCR đã sẵn sàng!


In [ ]:
# Thực hiện OCR
result = ocr.ocr(gray, cls=True)

# In kết quả văn bản
for idx in range(len(result)):
    for line in result[idx]:
        print(f"Văn bản: {line[1][0]}   |   Độ tin cậy: {line[1][1]:.4f}")

Văn bản: Tre Viet Nam.   |   Độ tin cậy: 0.9691
Văn bản: Bao bung than boc lay than   |   Độ tin cậy: 0.9985
Văn bản: Tay om, tay niu tre gan nhau them..   |   Độ tin cậy: 0.9832
Văn bản: Thuong nhau, tre chang o rieng   |   Độ tin cậy: 0.9795
Văn bản: Huy thanh tur do ma nen hoi nguoi..   |   Độ tin cậy: 0.9634
Văn bản: "Chang may than gay canh roi.   |   Độ tin cậy: 0.9781
Văn bản: Van nguyen cai goc truyen doi cho mang..   |   Độ tin cậy: 0.9844
Văn bản: Noi tre dau chiu moc cong   |   Độ tin cậy: 0.9982
Văn bản: "Chua len da nhon nhu chong la thuong   |   Độ tin cậy: 0.9962
Văn bản: Hung tran phoi nang phoi suong.   |   Độ tin cậy: 0.9797
Văn bản: 'Co manh ao coc, tre nhuong cho con.   |   Độ tin cậy: 0.9833
Văn bản: Nguyen Duy "   |   Độ tin cậy: 0.9981
